# 12 — 3D raw images using channel–mass calibration from matching CSV

This workflow uses a matching `*_3dimage.csv` file to extract the real channel–mass calibration,
then applies that calibration to raw 3D FPD image layers.

Workflow:

```text
3D image CSV → channel–mass calibration
raw image layers → total channel spectrum
calibration + channel spectrum → mass spectrum
mass spectrum → peak identification → bins
bins → 3D ion volumes
```

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from pymagsims import Spectrum, SIMSVolume
from pymagsims.raw_image import SIMSRawImage
from pymagsims.isotopes import load_builtin_isotopes, filter_isotopes
from pymagsims.plotting import plot_ion_image, plot_ion_image_grid, plot_volume_slice, plot_array_slider, plot_array_grid_slider
from pymagsims.binning import merge_bins_by_element, mass_bins_to_channel_bins
from pymagsims.interactive import plot_volume_label_slider, plot_volume_slider

DATA = Path("../data")
RAW_LAYER_DIR = DATA / "3d"

CSV_3D_FILE = DATA / "202505077-MEMS-011_neg_500mT_3dimage.csv"
IMAGE_SHAPE = (256, 256)

## 1. Find raw image layer files with natural sorting

In [ ]:
def natural_sort_key(path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]

paths = sorted(RAW_LAYER_DIR.glob("*Image_*.raw"), key=natural_sort_key)

print(f"Found {len(paths)} raw image layers")
for p in paths:
    print(p.name)

## 2. Extract channel–mass calibration from the matching 3D-image CSV

In [ ]:
def extract_calibration_from_3d_csv(path, encoding="latin1", n_channels=12000):
    path = Path(path)

    with path.open("r", encoding=encoding) as f:
        lines = f.readlines()

    end_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("%end"):
            end_idx = i
            break

    if end_idx is None:
        raise ValueError("Could not find %end marker in CSV file.")

    spectrum_rows = []
    spectrum_start = end_idx + 1

    for line in lines[spectrum_start:spectrum_start + n_channels]:
        parts = [p.strip() for p in line.strip().split(";") if p.strip() != ""]
        if len(parts) >= 3:
            spectrum_rows.append(parts[:3])

    spectrum_df = pd.DataFrame(spectrum_rows, columns=["Channel", "Mass", "Amplitude"])
    spectrum_df = spectrum_df.apply(pd.to_numeric, errors="coerce").dropna()
    spectrum_df["Channel"] = spectrum_df["Channel"].astype(int)

    calibration_df = spectrum_df[["Channel", "Mass"]].copy()
    return spectrum_df, calibration_df

csv_spectrum_df, calibration_df = extract_calibration_from_3d_csv(CSV_3D_FILE)

display(csv_spectrum_df.head())
display(calibration_df.head())
display(calibration_df.tail())

print("Channel range:", calibration_df["Channel"].min(), "to", calibration_df["Channel"].max())
print("Mass range:", calibration_df["Mass"].min(), "to", calibration_df["Mass"].max())

## 3. Save calibration CSV for reuse

In [ ]:
calibration_dir = DATA / "calibration"
calibration_dir.mkdir(parents=True, exist_ok=True)

calibration_file = calibration_dir / "neg_500mT_channel_mass_calibration.csv"
calibration_df.to_csv(calibration_file, index=False)

print(f"Saved calibration to: {calibration_file}")

## 4. Plot calibration curve

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=calibration_df["Channel"],
        y=calibration_df["Mass"],
        mode="lines",
        name="Calibration",
        hovertemplate="Channel: %{x}<br>Mass: %{y:.5f} amu<extra></extra>",
    )
)
fig.update_layout(
    title="Channel–mass calibration",
    xaxis_title="Detector channel",
    yaxis_title="Mass / amu",
    template="plotly_white",
)
fig

## 5. Build total counts vs detector channel from raw image layers

In [ ]:
all_counts = []

for path in paths:
    raw = SIMSRawImage.from_fpd_raw(path, shape=IMAGE_SHAPE)
    counts = raw.events["Channel"].value_counts()
    all_counts.append(counts)

channel_counts = (
    pd.concat(all_counts, axis=1)
    .fillna(0)
    .sum(axis=1)
    .sort_index()
)

channel_spectrum = pd.DataFrame(
    {
        "Channel": channel_counts.index.astype(int),
        "Counts": channel_counts.values.astype(int),
    }
)

display(channel_spectrum.head())
display(channel_spectrum.tail())
print("Total counts:", channel_spectrum["Counts"].sum())

## 6. Plot total counts vs detector channel

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "iframe"

pio.renderers.default = "notebook_connected"  # or try "iframe" / "browser" / "vscode"

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=channel_spectrum["Channel"],
        y=channel_spectrum["Counts"],
        mode="lines",
        name="Total counts",
        hovertemplate="Channel: %{x}<br>Counts: %{y}<extra></extra>",
    )
)

fig.update_layout(
    title="Total counts vs detector channel",
    xaxis_title="Detector channel",
    yaxis_title="Total counts",
    template="plotly_white",
    hovermode="closest",
)

fig.update_yaxes(type="log")

fig.show()

## 7. Convert channel spectrum to calibrated mass spectrum

In [ ]:
calibrated_spectrum_df = calibration_df.merge(
    channel_spectrum.rename(columns={"Counts": "Amplitude"}),
    on="Channel",
    how="left",
)

calibrated_spectrum_df["Amplitude"] = calibrated_spectrum_df["Amplitude"].fillna(0)

spec = Spectrum(
    data=calibrated_spectrum_df[["Channel", "Mass", "Amplitude"]],
    metadata={
        "source": "3D raw image layers with calibration extracted from matching CSV",
        "calibration_file": str(calibration_file),
    },
    name="3d_raw_image_calibrated_spectrum",
)

spec.plot(x="Mass", y="Amplitude", log_y=True);

In [ ]:
isotopes = load_builtin_isotopes(max_atomic_number=190)

isotopes_relevant = filter_isotopes(
    isotopes,
    min_abundance=0.5,
)

In [ ]:
spec.plot_with_element_markers(
    isotope_table=isotopes,
    elements=["Si", "Ga", "O", "Cr", "Au", "N", "O"],
    log_y=True,
    xlim=(0, 270),
    min_abundance=0.1,
);

## 8. Interactive calibrated mass spectrum

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=spec.data["Mass"],
        y=spec.data["Amplitude"],
        mode="lines",
        name="Calibrated spectrum",
        hovertemplate="Mass: %{x:.5f} amu<br>Counts: %{y}<extra></extra>",
    )
)
fig.update_layout(
    title="Calibrated total spectrum from 3D raw image layers",
    xaxis_title="Mass / amu",
    yaxis_title="Total counts",
    template="plotly_white",
    hovermode="closest",
)
fig.update_yaxes(type="log")
fig

## 9. Automatic peak detection and isotope assignment

In [ ]:
assignments = spec.assign_peaks(
    isotope_table=isotopes_relevant,
    tolerance=1,
    prominence=40,
    distance=5,
)


pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

#pd.set_option("display.max_rows", None)
#pd.set_option("display.max_columns", None)
#pd.set_option("display.width", None)

display(assignments)
print(f"Number of candidate assignments: {len(assignments)}")




In [ ]:
# only one element
assignments[assignments["element"] == "Cr"]

In [ ]:
#assignments_sorted = assignments.sort_values(
#    ["measured_mass", "abs_mass_error"]
#)
#
#display(assignments_sorted)


In [ ]:
#exact_mass = pd.to_numeric(assignments["exact_mass"], errors="coerce")

#filtered = assignments.loc[
#    (exact_mass >= 100) &
#    (exact_mass <= 200)
#]

#display(filtered)

## 10. Plot spectrum with peaks using Plotly

In [ ]:
from pymagsims.interactive import plot_spectrum_with_peaks_interactive

fig, assignments = plot_spectrum_with_peaks_interactive(
    spec,
    isotope_table=isotopes_relevant,
    tolerance=1,
    prominence=40,
    distance=5,
    log_y=True,
    label_peaks=True,
)

fig

## 11. Create and save mass bins

In [ ]:
mass_bins = spec.create_bins_from_assignments(assignments, width=0.3)

display(mass_bins)

bin_dir = DATA / "bins"
bin_dir.mkdir(parents=True, exist_ok=True)

mass_bin_file = bin_dir / "mass_bins_from_3d_csv_calibration.csv"
mass_bins.to_csv(mass_bin_file, index=False)

print(f"Saved mass bins to: {mass_bin_file}")

## 12. Convert mass bins to channel bins

In [ ]:
converted_bins = mass_bins_to_channel_bins(
    mass_bins,
    calibration_df,
)

display(converted_bins)

channel_bin_file = bin_dir / "channel_bins_from_3d_csv_calibration.csv"
converted_bins.to_csv(channel_bin_file, index=False)

print(f"Saved channel bins to: {channel_bin_file}")

## 13. Select bins for 3D reconstruction

In [ ]:
#selected_bins = converted_bins.head(5).copy()
selected_bins = converted_bins
display(selected_bins[["label", "mass_min", "mass_max", "ch_min", "ch_max"]])

In [ ]:
#element_bins = merge_bins_by_element(converted_bins)
#display(element_bins)

In [ ]:
#selected_bins = element_bins.copy()
#selected_bins

## 14. Build 3D ion volumes

In [ ]:
volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    bins=selected_bins,
    spectrum=None,
    include_total=True,
    shape=IMAGE_SHAPE,
)

print(volume.labels())
display(volume.metadata)

for label, arr in volume.volumes.items():
    print(f"{label}: shape={arr.shape}, total_counts={arr.sum()}")

## 15. Plot one slice

In [ ]:
plot_volume_slice(volume, label="Total", z=0, log=True, cmap="viridis");

In [ ]:
label = [l for l in volume.labels() if l != "Total"][0]
plot_volume_slice(volume, label=label, z=0, log=True, cmap="magma");

## 16. Plot summed projections

In [ ]:
projection_images = {
    label: volume.sum_projection(label)
    for label in volume.labels()
}

plot_ion_image_grid(
    projection_images,
    log=True,
    ncols=3,
    cmaps=["gray", "viridis", "magma", "plasma", "cividis", "inferno", "turbo"],
);

## 17. Depth profiles from the 3D volume

In [ ]:
profiles = []

for label in volume.labels():
    profile = volume.depth_profile(label)
    profile = profile.rename(columns={label: "Intensity"})
    profile["Label"] = label
    profiles.append(profile)

depth_profiles = pd.concat(profiles, ignore_index=True)
display(depth_profiles.head())

fig, ax = plt.subplots(figsize=(8, 4))

for label, group in depth_profiles.groupby("Label"):
    ax.plot(group["Slice"], group["Intensity"], marker="o", linewidth=1, label=label)

ax.set_xlabel("Slice / layer")
ax.set_ylabel("Integrated counts")
ax.set_yscale("log")
ax.grid(True)
ax.legend()
fig.tight_layout()

## 18. Plotly layer slider

In [ ]:
plot_volume_slider(volume, label="Total", log=True)

In [ ]:
#label = [l for l in volume.labels() if l != "Total"][11]
#plot_volume_slider(volume, label=label, log=True, colorscale="Magma")

#matches = [label for label in volume.labels() if "Si" in label]
#print(matches)

plot_volume_slider(
    volume,
    label="28Si",
    log=True,
    colorscale="Magma",
)

In [ ]:
# 3. Sum already-reconstructed bins by element
si_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Si",
)

hf_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Hf",
)

ti_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Ti",
)

#cr_volume = volume.summed_by_element(
#    bins=selected_bins,
#    element="Cr",
#)

o_volume = volume.summed_by_element(
    bins=selected_bins,
    element="O",
)

#n_volume = volume.summed_by_element(
#    bins=selected_bins,
#    element="N",
#)

gd_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Gd",
)

hf_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Hf",
)

ge_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Ge",
)

mg_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Mg",
)

cl_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Cl",
)

ga_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Ga",
)

se_volume = volume.summed_by_element(
    bins=selected_bins,
    element="Se",
)

In [ ]:
plot_ion_image(
    si_volume.sum(axis=0),
    log=True,
    title="Si summed isotope map",
)

In [ ]:
plot_array_slider(
    si_volume,
    label="Si summed isotope bins",
    log=True,
    colorscale="Magma",
)

In [ ]:
sorted(selected_bins["element"].dropna().unique())

In [ ]:
arrays = {
    "Total": volume.get("Total"),
    "Si": volume.summed_by_element(selected_bins, "Si"),
    "O": volume.summed_by_element(selected_bins, "O"),
    "Ti": volume.summed_by_element(selected_bins, "Ti"),
    "Ge": volume.summed_by_element(selected_bins, "Ge"),
    "Mg": volume.summed_by_element(selected_bins, "Mg"),
    "Se": volume.summed_by_element(selected_bins, "Se"),
    "Ga": volume.summed_by_element(selected_bins, "Ga"),
    "Hf": volume.summed_by_element(selected_bins, "Hf"),
}

plot_array_grid_slider(
    arrays,
    log=True,
    nrows=3,
    ncols=3,
    colorscale={
        "Total": "Gray",
        "Si": "Teal",
        "O": "Viridis",
        "Ti": "Cividis",
        "Ge": "Electric",
        "Mg": "Purp",
        "Se": "Hot",
        "Ga": "Jet",
        "Hf": "Dense",
    },
)

In [ ]:
si_proj = si_volume.sum(axis=0)
o_proj = o_volume.sum(axis=0)
ti_proj = hf_volume.sum(axis=0)
ge_proj = si_volume.sum(axis=0)
mg_proj = mg_volume.sum(axis=0)
se_proj = se_volume.sum(axis=0)
ga_proj = ga_volume.sum(axis=0)
hf_proj = hf_volume.sum(axis=0)



In [ ]:
from pymagsims.plotting import plot_rgb_overlay

plot_rgb_overlay(
    red=si_proj,
    green=ti_proj,
    blue=hf_proj,
    red_label="Ti",
    green_label="Si",
    blue_label="Mg",
)

In [ ]:
from pymagsims.plotting import plot_rgb_overlay_slider

plot_rgb_overlay_slider(
    red=volume.get("Total"),
    green=si_volume,
    blue=ti_volume,
    red_label="Total",
    green_label="Si",
    blue_label="Ti",
    log=True,
)

In [ ]:
from pymagsims.plotting import plot_element_overlay_slider

plot_element_overlay_slider(
    background=volume.get("Total"),
    overlays={
        "Ti": ti_volume,
        "Si": si_volume,
        "Mg": mg_volume,
    },
    colors={
        "Ti": (1.0, 0.2, 1.0),
        "Si": (0.0, 0.7, 1.0),
        "Mg": (1.0, 0.0, 1.0),
    },
    alpha=0.5,
    background_scale=0.9,
    background_gamma=1.2,
    overlay_gamma=0.5,
    log=True,
)

In [ ]:
from pymagsims.export import export_imagej_hyperstack

arrays = {
    "Total": volume.get("Total"),
    "Ti": ti_volume,
    "Si": si_volume,
    "Mg": mg_volume,
}

for k, v in arrays.items():
    print(k, v.sum(), v.max())

export_imagej_hyperstack(
    arrays,
    "../data/export/ion_hyperstack.tif",
)

In [ ]:
# Export as vti for paraview
from pymagsims.export import export_paraview_vti

arrays = {
    "Total": volume.get("Total"),
    "Ti": ti_volume,
    "Si": si_volume,
    "Mg": mg_volume,
}

export_paraview_vti(
    arrays,
    "../data/export/sims_volume.vti",
    spacing=(1.0, 1.0, 1.0),
)

In [ ]:
# Export one file per element
for label, arr in arrays.items():
    export_paraview_vti(
        {label: arr},
        f"../data/export/{label}.vti",
    )

## Summary

This notebook extracts a real channel–mass calibration from the matching 3D-image CSV and uses it for calibrated peak/binned 3D ion volume reconstruction.